In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
bronze_payments_df = spark.table(
    "ecommerce_lakehouse.bronze.payments_raw"
)

In [0]:
silver_payments_df = bronze_payments_df \
    .dropDuplicates([
        "order_id",
        "payment_sequential"
    ]) \
    .withColumn(
        "payment_type",
        initcap(col("payment_type"))
    )

In [0]:
silver_payments_df = silver_payments_df.withColumn(
    "installment_category",
    when(col("payment_installments") == 1, "Single Payment")
    .when(col("payment_installments") <= 6, "Short Term")
    .otherwise("Long Term")
)

In [0]:
silver_payments_df = silver_payments_df.withColumn(
    "is_high_value_payment",
    when(col("payment_value") > 500, True)
    .otherwise(False)
)

In [0]:
(
    silver_payments_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(
            "ecommerce_lakehouse.silver.payments_clean"
        )
)

In [0]:
%sql
SELECT *
FROM ecommerce_lakehouse.silver.payments_clean
LIMIT 10;